# StartupMatch: Investor-Startup Matching Model

This notebook contains the ML model for matching investors with startups.
You can run this on Kaggle for GPU acceleration.

## Model Architecture
- **Approach**: Gradient Boosting + Neural Network hybrid
- **Features**: Sector, Stage, Location, Funding alignment scores
- **Training**: Supervised learning with labeled match data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load and Prepare Data

In [ ]:
# Load your CSV data
# Update paths to point to your data files

DATA_PATH = '../backend/data/'

investors = pd.read_csv(DATA_PATH + 'investors.csv')
startups = pd.read_csv(DATA_PATH + 'startups.csv')
matches = pd.read_csv(DATA_PATH + 'matches.csv')

print("Investors:", len(investors))
print("Startups:", len(startups))
print("Matches:", len(matches))

## 2. Feature Engineering

Create features from investor and startup profiles.

In [ ]:
def create_match_features(investor, startup):
    """Create features for a single investor-startup pair"""
    features = {}
    
    # Sector match
    investor_sectors = str(investor.get('sectors', '')).lower().split(',')
    startup_sector = str(startup.get('sector', '')).lower()
    features['sector_match'] = 1.0 if startup_sector in investor_sectors else 0.0
    
    # Stage match
    stage_order = ['pre-seed', 'seed', 'series a', 'series b', 'series c', 'growth']
    inv_stage = str(investor.get('stage_preference', '')).lower()
    start_stage = str(startup.get('stage', '')).lower()
    
    try:
        if inv_stage in stage_order and start_stage in stage_order:
            idx_diff = abs(stage_order.index(inv_stage) - stage_order.index(start_stage))
            features['stage_match'] = max(0, 1 - idx_diff * 0.25)
        else:
            features['stage_match'] = 0.5
    except:
        features['stage_match'] = 0.5
    
    # Location match
    investor_locs = str(investor.get('locations', '')).lower().split(',')
    startup_loc = str(startup.get('location', '')).lower()
    features['location_match'] = 1.0 if 'global' in investor_locs or startup_loc in investor_locs else 0.3
    
    # Funding match
    min_check = float(investor.get('min_check', 0) or 0)
    max_check = float(investor.get('max_check', 0) or 0)
    funding_needed = float(startup.get('funding_needed', 0) or 0)
    
    if funding_needed == 0:
        features['funding_match'] = 0.5
    elif min_check <= funding_needed <= max_check:
        features['funding_match'] = 1.0
    else:
        features['funding_match'] = min(max_check / max(funding_needed, 1), funding_needed / max(min_check, 1))
    
    # Numeric features
    features['team_size'] = float(startup.get('team_size', 0) or 0)
    features['revenue'] = float(startup.get('revenue', 0) or 0)
    features['growth_rate'] = float(startup.get('growth_rate', 0) or 0)
    
    return features

# Create training dataset
print("Creating feature dataset...")
feature_list = []
labels = []

for _, match in matches.iterrows():
    investor = investors[investors['id'] == match['investor_id']]
    startup = startups[startups['id'] == match['startup_id']]
    
    if len(investor) > 0 and len(startup) > 0:
        features = create_match_features(investor.iloc[0].to_dict(), startup.iloc[0].to_dict())
        feature_list.append(features)
        # Label: 1 if score > 0.7, else 0 (you can adjust threshold)
        labels.append(1 if match['score'] > 0.7 else 0)

X = pd.DataFrame(feature_list)
y = np.array(labels)

print(f"Dataset created: {len(X)} samples")
print(f"Features: {list(X.columns)}")
print(f"Positive samples: {sum(y)}, Negative samples: {len(y) - sum(y)}")

## 3. Train Gradient Boosting Model (Baseline)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

# Evaluate
train_acc = gb_model.score(X_train_scaled, y_train)
test_acc = gb_model.score(X_test_scaled, y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Feature importance
print("\nFeature Importance:")
for feat, imp in sorted(zip(X.columns, gb_model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.4f}")

## 4. Neural Network Model (PyTorch)

In [ ]:
class MatchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class MatchingNN(nn.Module):
    def __init__(self, input_dim):
        super(MatchingNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# Create data loaders
train_dataset = MatchDataset(X_train_scaled, y_train)
test_dataset = MatchDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

nn_model = MatchingNN(X_train_scaled.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)

print(f"Model architecture:\n{nn_model}")

In [ ]:
# Training loop
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predictions = (outputs > 0.5).float()
        correct += (predictions == y_batch).sum().item()
        total += len(y_batch)
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item()
            predictions = (outputs > 0.5).float()
            correct += (predictions == y_batch).sum().item()
            total += len(y_batch)
    
    return total_loss / len(loader), correct / total

# Train
epochs = 50
best_acc = 0

print("Training Neural Network...")
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(nn_model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(nn_model, test_loader, criterion, device)
    
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(nn_model.state_dict(), 'best_model.pth')
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nBest Test Accuracy: {best_acc:.4f}")

## 5. Make Predictions

In [ ]:
# Load best model
nn_model.load_state_dict(torch.load('best_model.pth'))
nn_model.eval()

def predict_match(investor, startup):
    """Predict match probability for a single investor-startup pair"""
    features = create_match_features(investor, startup)
    feature_vector = pd.DataFrame([features])
    
    # Scale
    feature_scaled = scaler.transform(feature_vector)
    feature_tensor = torch.FloatTensor(feature_scaled).to(device)
    
    # Predict
    with torch.no_grad():
        prob = nn_model(feature_tensor).item()
    
    return prob

# Example: Test with first investor and startup
if len(investors) > 0 and len(startups) > 0:
    test_inv = investors.iloc[0].to_dict()
    test_start = startups.iloc[0].to_dict()
    
    prob = predict_match(test_inv, test_start)
    print(f"Match probability: {prob:.4f}")
    print(f"Prediction: {'Match' if prob > 0.5 else 'No Match'}")

## 6. Save Model & Scaler

In [ ]:
import pickle

# Save models
torch.save(nn_model.state_dict(), 'matching_nn.pth')
torch.save(gb_model, 'matching_gb.pkl')
pickle.dump(scaler, open('scaler.pkl', 'wb'))

print("Models saved:")
print("  - matching_nn.pth (PyTorch)")
print("  - matching_gb.pkl (Gradient Boosting)")
print("  - scaler.pkl")

## 7. How to Train with New Data

When you acquire labeled data (successful matches), follow these steps:

1. **Add labeled matches** to your CSV:
   - Create `labeled_matches.csv` with: `investor_id`, `startup_id`, `success` (1/0)
   - `success=1` means the match resulted in investment
   
2. **Re-train the model**:
   ```python
   labeled_data = pd.read_csv('labeled_matches.csv')
   # Use 'success' column as y label instead of score threshold
   ```

3. **Update prediction threshold** based on precision/recall trade-off

4. **Re-train periodically** as you get more data

In [ ]:
# Training function for labeled data
def train_with_labeled_data(labeled_csv_path, model_path='matching_nn.pth'):
    """
    Train model with labeled match data.
    
    labeled_csv should have columns:
    - investor_id: ID of investor
    - startup_id: ID of startup  
    - success: 1 if match was successful, 0 otherwise
    """
    
    # Load labeled data
    labeled = pd.read_csv(labeled_csv_path)
    
    # Create features for each labeled pair
    feature_list = []
    labels = []
    
    for _, row in labeled.iterrows():
        investor = investors[investors['id'] == row['investor_id']]
        startup = startups[startups['id'] == row['startup_id']]
        
        if len(investor) > 0 and len(startup) > 0:
            features = create_match_features(investor.iloc[0].to_dict(), startup.iloc[0].to_dict())
            feature_list.append(features)
            labels.append(row['success'])
    
    X = pd.DataFrame(feature_list)
    y = np.array(labels)
    
    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train
    # ... (same training loop as above)
    
    print(f"Trained on {len(X)} labeled samples")
    return model

# Uncomment to use:
# train_with_labeled_data('labeled_matches.csv')

---
## Usage Guide

### Running on Kaggle:
1. Upload this notebook to Kaggle
2. Upload your CSV data files to /kaggle/input/
3. Update DATA_PATH to point to your data
4. Run all cells
5. Download trained models (.pth files)

### Integration with Backend:
The trained model can be called from Flask API:
```python
# In backend/matching.py, add:
import torch
nn_model = MatchingNN(input_dim)
nn_model.load_state_dict(torch.load('matching_nn.pth'))
```

### Getting Better Results:
- Collect more labeled data (successful matches)
- Add more features (team experience, market size, etc.)
- Try different architectures (transformers, attention)
- Ensemble multiple models